# 공공 통계 원천 갱신일 확인 자동화

index.go.kr 4개 지표와 KOSIS 12개 통계표의 갱신일을 확인합니다.
CSV에는 지표별 최신 확인 상태를 저장합니다. 과거 이력 누적 방식이 아닙니다.
회사 내부 DB 적재 완료나 데이터 값의 정확성은 검사하지 않습니다.

README에 따라 패키지와 환경변수를 준비한 뒤 순서대로 실행하세요.
공개용 사본에서는 인증키와 실행 출력을 제거했습니다. 외부 API 실행 검증 전입니다.


In [ ]:
import os
import csv
import re
import sys
from datetime import datetime
from pathlib import Path

import requests
from bs4 import BeautifulSoup

KOSIS_API_KEY = os.environ.get("KOSIS_API_KEY", "").strip()
if not KOSIS_API_KEY:
    raise RuntimeError("KOSIS_API_KEY 환경변수를 설정하고 커널을 재시작하세요.")


In [ ]:
BASE_URL = "https://www.index.go.kr/unity/potal/main/EachDtlPageDetail.do"
KOSIS_META_URL = "https://kosis.kr/openapi/statisticsData.do"
CSV_PATH = Path("index_update_log.csv")

# index.go.kr 지표 - (지표명, idx_cd)
INDEX_TARGETS = [
    ("통계청_경기종합지수", "1057"),
    ("통계청_소비자물가지수", "1060"),
    ("통계청_소비자동향지수(CSI)", "1058"),
    ("E나라지표_기업경기실사지수(BSI) 등", "1059"),
]

# KOSIS 통계표 - (지표명, orgId, tblId)
# orgId/tblId는 통계표 URL의 orgId, tblId 값과 동일하게 넣으면 됨 (vw_cd/list_id는 필요 없음)
KOSIS_TARGETS = [
    ("통계청_생활물가지수", "101", "DT_1J22005"),
    ("통계청_연령별 경제활동인구 총괄", "101", "DT_1DA7002S"),
    ("통계청_가구주의 성/연령별 추계가구-시도", "101", "DT_1BZ05041"),
    ("통계청_시나리오별 추계인구(전국)", "101", "DT_1BPA402"),
    ("통계청_가구원수별 가구당 월평균 가계수지", "101", "DT_1L9U005"),
    ("통계청_가계신용(용도별, 분기)", "301", "DT_151Y004"),
    ("통계청_대출행태서베이(대출수요)", "301", "DT_514Y003"),
    ("통계청_주요 인구지표(성비,인구성장률,인구구조,부양비 등)", "101", "DT_1BPA002"),
    ("통계청_연령 및 성별 인구-읍면동", "101", "DT_1IN1503"),
    ("통계청_인구,가구 및 주택-읍면동/시군구", "101", "DT_1IN1502"),
    ("통계청_조세부담률(GDP 대비 조세부담현황)", "110", "TX_11007_A783"),
    ("통계청_도로종류별 차종별 평균일교통량과 구성비", "116", "DT_MLTM_1240"),
]

DATE_PATTERN = re.compile(
    r"최근\s*갱신일\s*[:：]\s*([0-9]{4}-[0-9]{2}-[0-9]{2})"
    r"\s*\(\s*입력예정일\s*[:：]\s*([0-9]{4}-[0-9]{2}-[0-9]{2})\s*\)"
)

In [ ]:
def fetch_dates(idx_cd: str) -> dict:
    """index.go.kr 지표 페이지에서 최근갱신일/입력예정일 추출"""
    resp = requests.get(BASE_URL, params={"idx_cd": idx_cd}, timeout=15)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding

    soup = BeautifulSoup(resp.text, "html.parser")
    text = soup.get_text(" ", strip=True)

    match = DATE_PATTERN.search(text)
    if not match:
        return {"last_updated": None, "next_due": None, "raw_found": False}

    return {
        "last_updated": match.group(1),
        "next_due": match.group(2),
        "raw_found": True,
    }


def fetch_kosis_update_date(org_id: str, tbl_id: str) -> dict:
    """KOSIS Open API(getMeta, type=NCD)로 통계표의 자료갱신일(SEND_DE) 조회.
    NCD는 수록기간별 SEND_DE 목록을 주므로 그중 최댓값을 최근 갱신일로 사용."""
    params = {
        "method": "getMeta",
        "apiKey": KOSIS_API_KEY,
        "type": "NCD",
        "orgId": org_id,
        "tblId": tbl_id,
        "format": "json",
        "jsonVD": "Y",
    }
    resp = requests.get(KOSIS_META_URL, params=params, timeout=15)
    resp.raise_for_status()
    items = resp.json()

    if isinstance(items, dict):
        # 에러 응답은 {"err": "...", "errMsg": "..."} 형태로 내려옴
        return {"last_updated": None, "raw_found": False, "error": items}

    send_des = [it.get("SEND_DE") for it in items if it.get("SEND_DE")]
    if not send_des:
        return {"last_updated": None, "raw_found": False}

    return {"last_updated": max(send_des), "raw_found": True}

In [ ]:
def main():
    checked_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    fieldnames = ["확인시각", "출처", "키", "지표명", "최근갱신일", "입력예정일", "상태"]

    # 기존 CSV를 읽어 (출처,키) -> row 로 보관 (지표별 최신 상태만 유지)
    existing = {}
    if CSV_PATH.exists():
        with open(CSV_PATH, "r", newline="", encoding="utf-8-sig") as f:
            for row in csv.DictReader(f):
                existing[(row["출처"], row["키"])] = row

    # 1) index.go.kr 지표
    for name, idx_cd in INDEX_TARGETS:
        key = ("index.go.kr", idx_cd)
        try:
            info = fetch_dates(idx_cd)
        except requests.RequestException as e:
            print(f"[ERROR] {name} (idx_cd={idx_cd}): 요청 실패 - {type(e).__name__}", file=sys.stderr)
            existing[key] = {
                "확인시각": checked_at, "출처": "index.go.kr", "키": idx_cd, "지표명": name,
                "최근갱신일": "", "입력예정일": "", "상태": "ERROR",
            }
            continue

        if not info["raw_found"]:
            print(f"[WARN] {name} (idx_cd={idx_cd}): 날짜 패턴을 찾지 못함 (페이지 구조 변경 가능성)")
            existing[key] = {
                "확인시각": checked_at, "출처": "index.go.kr", "키": idx_cd, "지표명": name,
                "최근갱신일": "", "입력예정일": "", "상태": "NOT_FOUND",
            }
            continue

        print(
            f"[OK] {name} (idx_cd={idx_cd}): "
            f"최근 갱신일={info['last_updated']}, 입력예정일={info['next_due']}"
        )
        existing[key] = {
            "확인시각": checked_at, "출처": "index.go.kr", "키": idx_cd, "지표명": name,
            "최근갱신일": info["last_updated"], "입력예정일": info["next_due"], "상태": "OK",
        }

    # 2) KOSIS 통계표
    for name, org_id, tbl_id in KOSIS_TARGETS:
        key = ("KOSIS", tbl_id)
        try:
            info = fetch_kosis_update_date(org_id, tbl_id)
        except requests.RequestException as e:
            print(f"[ERROR] {name} (tblId={tbl_id}): 요청 실패 - {type(e).__name__}", file=sys.stderr)
            existing[key] = {
                "확인시각": checked_at, "출처": "KOSIS", "키": tbl_id, "지표명": name,
                "최근갱신일": "", "입력예정일": "", "상태": "ERROR",
            }
            continue

        if not info["raw_found"]:
            print(f"[WARN] {name} (tblId={tbl_id}): 자료갱신일을 찾지 못함 - API 오류 또는 갱신일 누락")
            existing[key] = {
                "확인시각": checked_at, "출처": "KOSIS", "키": tbl_id, "지표명": name,
                "최근갱신일": "", "입력예정일": "", "상태": "NOT_FOUND",
            }
            continue

        print(f"[OK] {name} (tblId={tbl_id}): 자료갱신일={info['last_updated']}")
        existing[key] = {
            "확인시각": checked_at, "출처": "KOSIS", "키": tbl_id, "지표명": name,
            "최근갱신일": info["last_updated"], "입력예정일": "", "상태": "OK",
        }

    # (출처,키)별 최신 상태만 남겨서 전체 재작성 (과거 기록 누적 방지)
    with open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(existing.values())

    print(f"\n결과가 갱신됨 (지표별 최신 상태만 유지): {CSV_PATH.resolve()}")


main()